In [ ]:
!pip -q install -U datasets transformers scikit-learn pandas

In [ ]:
import random
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from sklearn.model_selection import train_test_split

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


### Load the Dataset

In [ ]:
import json
import requests
import pandas as pd

BASE_URL = (
    "https://huggingface.co/datasets/Coldog2333/tiage"
    "/resolve/main/"
)

def load_tiage_split(split):
    url = BASE_URL + f"{split}.json"

    response = requests.get(url, timeout=60)
    response.raise_for_status()

    data = response.json()["dial_data"]["tiage"]
    return data

train_dialogues = load_tiage_split("train")
val_dialogues = load_tiage_split("validation")
test_dialogues = load_tiage_split("test")

print("Train dialogues:", len(train_dialogues))
print("Validation dialogues:", len(val_dialogues))
print("Test dialogues:", len(test_dialogues))

# Inspect the actual structure
print("\nFirst dialogue:")
print(json.dumps(train_dialogues[0], indent=2)[:3000])

In [ ]:
def create_examples(dialogues, history_turns=3):
    examples = []

    for dialogue in dialogues:
        turns = dialogue["turns"]

        for i in range(1, len(turns)):
            previous_topic = turns[i - 1]["topic_id"]
            current_topic = turns[i]["topic_id"]

            # 1 = topic shift, 0 = continuation
            label = int(current_topic != previous_topic)

            start = max(0, i - history_turns)

            history = "\n".join(
                f'{turns[j]["role"]}: {turns[j]["utterance"]}'
                for j in range(start, i)
            )

            current = turns[i]["utterance"]

            text = (
                f"Previous conversation:\n{history}\n\n"
                f"Current utterance:\n{current}"
            )

            examples.append({
                "dial_id": dialogue["dial_id"],
                "text": text,
                "label": label,
                "current_utterance": current
            })

    return pd.DataFrame(examples)


train_all = create_examples(train_dialogues)
val_all = create_examples(val_dialogues)

print("Training examples:", len(train_all))
print("Validation examples:", len(val_all))

print("\nTraining labels:")
print(train_all["label"].value_counts())

print("\nValidation labels:")
print(val_all["label"].value_counts())

Training examples: 4392
Validation examples: 1446

Training labels:
label
0    3505
1     887
Name: count, dtype: int64

Validation labels:
label
0    1135
1     311
Name: count, dtype: int64


In [ ]:
def sample_examples(df, n, seed=42):
    """Sample examples while preserving class proportions when possible."""
    if len(df) <= n:
        return df.reset_index(drop=True)

    if df["label"].nunique() == 2 and df["label"].value_counts().min() >= 2:
        sampled, _ = train_test_split(
            df,
            train_size=n,
            stratify=df["label"],
            random_state=seed
        )
        return sampled.reset_index(drop=True)

    return df.sample(n=n, random_state=seed).reset_index(drop=True)


train_df = sample_examples(train_all, 800, SEED)
val_df = sample_examples(val_all, 200, SEED)

print("\nSelected training examples:", len(train_df))
print("Selected validation examples:", len(val_df))

print("\nTraining distribution:")
print(train_df["label"].value_counts())

print("\nValidation distribution:")
print(val_df["label"].value_counts())

print("\nSample example:")
print("Text:\n", train_df.iloc[0]["text"])
print("Label:", train_df.iloc[0]["label"])


Selected training examples: 800
Selected validation examples: 200

Training distribution:
label
0    638
1    162
Name: count, dtype: int64

Validation distribution:
label
0    157
1     43
Name: count, dtype: int64

Sample example:
Text:
 Previous conversation:
user: hey , what are you planning for today ?

Current utterance:
oh not much , my name is heather and just looking to chat with people
Label: 0


### Train the Model

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

MODEL_NAME = "bert-base-uncased"
MAX_LENGTH = 192
BATCH_SIZE = 32
EPOCHS = 100
LEARNING_RATE = 2e-5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
).to(device)


class DialogueDataset(Dataset):
    def __init__(self, dataframe):
        self.texts = dataframe["text"].tolist()
        self.labels = dataframe["label"].astype(int).tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return {
            "text": self.texts[idx],
            "label": self.labels[idx]
        }


def collate_batch(batch):
    texts = [item["text"] for item in batch]
    labels = torch.tensor(
        [item["label"] for item in batch],
        dtype=torch.long
    )

    encodings = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )

    encodings["labels"] = labels
    return encodings


train_loader = DataLoader(
    DialogueDataset(train_df),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_batch
)

val_loader = DataLoader(
    DialogueDataset(val_df),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch
)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    for batch in train_loader:
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        optimizer.zero_grad()

        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / max(len(train_loader), 1)

    print(
        f"Epoch {epoch + 1}/{EPOCHS} "
        f"| Training loss: {average_loss:.4f}"
    )

Using device: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/100 | Training loss: 0.5105
Epoch 2/100 | Training loss: 0.4523
Epoch 3/100 | Training loss: 0.3851
Epoch 4/100 | Training loss: 0.2564
Epoch 5/100 | Training loss: 0.1473
Epoch 6/100 | Training loss: 0.0778
Epoch 7/100 | Training loss: 0.0563
Epoch 8/100 | Training loss: 0.0377
Epoch 9/100 | Training loss: 0.0310
Epoch 10/100 | Training loss: 0.0379
Epoch 11/100 | Training loss: 0.0318
Epoch 12/100 | Training loss: 0.0759
Epoch 13/100 | Training loss: 0.0358
Epoch 14/100 | Training loss: 0.0234
Epoch 15/100 | Training loss: 0.0212
Epoch 16/100 | Training loss: 0.0338
Epoch 17/100 | Training loss: 0.0155
Epoch 18/100 | Training loss: 0.0114
Epoch 19/100 | Training loss: 0.0103
Epoch 20/100 | Training loss: 0.0051
Epoch 21/100 | Training loss: 0.0023
Epoch 22/100 | Training loss: 0.0090
Epoch 23/100 | Training loss: 0.0024
Epoch 24/100 | Training loss: 0.0020
Epoch 25/100 | Training loss: 0.0080
Epoch 26/100 | Training loss: 0.0097
Epoch 27/100 | Training loss: 0.0072
Epoch 28/1

### Evaluate the model

In [ ]:
model.eval()

all_predictions = []
all_labels = []
all_probabilities = []

with torch.no_grad():
    for batch in val_loader:
        labels = batch["labels"].numpy()

        inputs = {
            key: value.to(device)
            for key, value in batch.items()
            if key != "labels"
        }

        outputs = model(**inputs)

        probabilities = torch.softmax(outputs.logits, dim=-1)
        predictions = torch.argmax(probabilities, dim=-1)

        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(labels)

        all_probabilities.extend(
            probabilities.cpu().numpy()
        )


print("\nAccuracy:", round(
    accuracy_score(all_labels, all_predictions), 4
))

print("Macro F1:", round(
    f1_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    ), 4
))

print("\nClassification report:")
print(classification_report(
    all_labels,
    all_predictions,
    labels=[0, 1],
    target_names=["CONTINUATION", "TOPIC_SHIFT"],
    zero_division=0
))

print("Confusion matrix:")
print(
    confusion_matrix(
        all_labels,
        all_predictions,
        labels=[0, 1]
    )
)


Accuracy: 0.86
Macro F1: 0.7576

Classification report:
              precision    recall  f1-score   support

CONTINUATION       0.87      0.96      0.92       157
 TOPIC_SHIFT       0.78      0.49      0.60        43

    accuracy                           0.86       200
   macro avg       0.83      0.73      0.76       200
weighted avg       0.85      0.86      0.85       200

Confusion matrix:
[[151   6]
 [ 22  21]]


In [ ]:
def predict_topic_shift(previous_turns, current_utterance):
    """
    previous_turns: list of preceding utterances, oldest first
    current_utterance: the new utterance to classify
    """
    history = "\n".join(previous_turns[-3:])

    text = (
        f"Previous conversation:\n{history}\n\n"
        f"Current utterance:\n{current_utterance}"
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH
    ).to(device)

    model.eval()

    with torch.no_grad():
        outputs = model(**inputs)
        probabilities = torch.softmax(outputs.logits, dim=-1)[0]

    predicted_label = int(torch.argmax(probabilities).item())

    return {
        "prediction": (
            "TOPIC_SHIFT"
            if predicted_label == 1
            else "CONTINUATION"
        ),
        "continuation_probability": round(
            probabilities[0].item(), 4
        ),
        "topic_shift_probability": round(
            probabilities[1].item(), 4
        )
    }


tests = [
    {
        "history": [
            "Show me the employee headcount by department.",
            "The engineering department has 120 employees."
        ],
        "current": "What about the sales department?"
    },
    {
        "history": [
            "Show me the employee headcount by department.",
            "The engineering department has 120 employees."
        ],
        "current": "How is the weather today?"
    },
    {
        "history": [
            "Tell me about dogs.",
            "Dogs make wonderful pets."
        ],
        "current": "What is the capital of France?"
    }
]

for test in tests:
    result = predict_topic_shift(
        test["history"],
        test["current"]
    )

    print("\nCurrent utterance:", test["current"])
    print(result)


Current utterance: What about the sales department?
{'prediction': 'TOPIC_SHIFT', 'continuation_probability': 0.0089, 'topic_shift_probability': 0.9911}

Current utterance: How is the weather today?
{'prediction': 'TOPIC_SHIFT', 'continuation_probability': 0.0005, 'topic_shift_probability': 0.9995}

Current utterance: What is the capital of France?
{'prediction': 'TOPIC_SHIFT', 'continuation_probability': 0.0017, 'topic_shift_probability': 0.9983}


In [ ]:
import os
import shutil
from google.colab import files

# Save the fine-tuned model and tokenizer
save_dir = "/content/bert_tiage_classifier"

model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

# Package everything into a ZIP file
zip_path = shutil.make_archive(
    "/content/bert_tiage_classifier",
    "zip",
    root_dir="/content",
    base_dir="bert_tiage_classifier"
)

print("Model saved and packaged:", zip_path)

# Download to your computer
files.download(zip_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved and packaged: /content/bert_tiage_classifier.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

# all_probabilities contains [P(continuation), P(topic_shift)]
probs = np.asarray(all_probabilities)
y_true = np.asarray(all_labels)

for threshold in [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]:
    y_pred = (probs[:, 1] >= threshold).astype(int)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    tn, fp, fn, tp = cm.ravel()

    precision = tp / (tp + fp) if tp + fp else 0
    recall = tp / (tp + fn) if tp + fn else 0
    accuracy = (tp + tn) / len(y_true)

    print(
        f"Threshold={threshold:.2f} | "
        f"Accuracy={accuracy:.3f} | "
        f"Shift precision={precision:.3f} | "
        f"Shift recall={recall:.3f} | "
        f"TP={tp}, FP={fp}, FN={fn}"
    )

Threshold=0.30 | Accuracy=0.860 | Shift precision=0.778 | Shift recall=0.488 | TP=21, FP=6, FN=22
Threshold=0.35 | Accuracy=0.860 | Shift precision=0.778 | Shift recall=0.488 | TP=21, FP=6, FN=22
Threshold=0.40 | Accuracy=0.860 | Shift precision=0.778 | Shift recall=0.488 | TP=21, FP=6, FN=22
Threshold=0.45 | Accuracy=0.860 | Shift precision=0.778 | Shift recall=0.488 | TP=21, FP=6, FN=22
Threshold=0.50 | Accuracy=0.860 | Shift precision=0.778 | Shift recall=0.488 | TP=21, FP=6, FN=22
Threshold=0.55 | Accuracy=0.860 | Shift precision=0.778 | Shift recall=0.488 | TP=21, FP=6, FN=22
Threshold=0.60 | Accuracy=0.860 | Shift precision=0.778 | Shift recall=0.488 | TP=21, FP=6, FN=22


### Weight class change

In [ ]:
import torch
import torch.nn as nn

# Count examples belonging to each class
counts = train_df["label"].value_counts().reindex(
    [0, 1],
    fill_value=0
)

# Give more weight to the minority class
class_weights = len(train_df) / (
    2 * torch.tensor(counts.values, dtype=torch.float32)
)

class_weights = class_weights.to(device)

print("Class counts:", counts.to_dict())
print("Class weights:", class_weights.tolist())

# Weighted cross-entropy loss
criterion = nn.CrossEntropyLoss(weight=class_weights)

Class counts: {0: 638, 1: 162}
Class weights: [0.6269592046737671, 2.4691357612609863]


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

MODEL_NAME = "bert-base-uncased"
MAX_LENGTH = 192
BATCH_SIZE = 32
EPOCHS = 100
LEARNING_RATE = 2e-5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
).to(device)


class DialogueDataset(Dataset):
    def __init__(self, dataframe):
        self.texts = dataframe["text"].tolist()
        self.labels = dataframe["label"].astype(int).tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return {
            "text": self.texts[idx],
            "label": self.labels[idx]
        }


def collate_batch(batch):
    texts = [item["text"] for item in batch]
    labels = torch.tensor(
        [item["label"] for item in batch],
        dtype=torch.long
    )

    encodings = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )

    encodings["labels"] = labels
    return encodings


train_loader = DataLoader(
    DialogueDataset(train_df),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_batch
)

val_loader = DataLoader(
    DialogueDataset(val_df),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch
)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    for batch in train_loader:
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        optimizer.zero_grad()

        labels = batch["labels"]

        inputs = {
            key: value
            for key, value in batch.items()
            if key != "labels"
        }

        outputs = model(**inputs)

        loss = criterion(
            outputs.logits,
            labels
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / max(len(train_loader), 1)

    print(
        f"Epoch {epoch + 1}/{EPOCHS} "
        f"| Training loss: {average_loss:.4f}"
    )

Using device: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/100 | Training loss: 0.6712
Epoch 2/100 | Training loss: 0.5900
Epoch 3/100 | Training loss: 0.5010
Epoch 4/100 | Training loss: 0.3557
Epoch 5/100 | Training loss: 0.2398
Epoch 6/100 | Training loss: 0.1782
Epoch 7/100 | Training loss: 0.1219
Epoch 8/100 | Training loss: 0.1082
Epoch 9/100 | Training loss: 0.0601
Epoch 10/100 | Training loss: 0.0334
Epoch 11/100 | Training loss: 0.0276
Epoch 12/100 | Training loss: 0.0254
Epoch 13/100 | Training loss: 0.0181
Epoch 14/100 | Training loss: 0.0148
Epoch 15/100 | Training loss: 0.0276
Epoch 16/100 | Training loss: 0.0400
Epoch 17/100 | Training loss: 0.0642
Epoch 18/100 | Training loss: 0.0550
Epoch 19/100 | Training loss: 0.0172
Epoch 20/100 | Training loss: 0.0260
Epoch 21/100 | Training loss: 0.0166
Epoch 22/100 | Training loss: 0.0144
Epoch 23/100 | Training loss: 0.0084
Epoch 24/100 | Training loss: 0.0092
Epoch 25/100 | Training loss: 0.0076
Epoch 26/100 | Training loss: 0.0063
Epoch 27/100 | Training loss: 0.0066
Epoch 28/1

In [ ]:
model.eval()

all_predictions = []
all_labels = []
all_probabilities = []

with torch.no_grad():
    for batch in val_loader:
        labels = batch["labels"].numpy()

        inputs = {
            key: value.to(device)
            for key, value in batch.items()
            if key != "labels"
        }

        outputs = model(**inputs)

        probabilities = torch.softmax(outputs.logits, dim=-1)
        predictions = torch.argmax(probabilities, dim=-1)

        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(labels)

        all_probabilities.extend(
            probabilities.cpu().numpy()
        )


print("\nAccuracy:", round(
    accuracy_score(all_labels, all_predictions), 4
))

print("Macro F1:", round(
    f1_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    ), 4
))

print("\nClassification report:")
print(classification_report(
    all_labels,
    all_predictions,
    labels=[0, 1],
    target_names=["CONTINUATION", "TOPIC_SHIFT"],
    zero_division=0
))

print("Confusion matrix:")
print(
    confusion_matrix(
        all_labels,
        all_predictions,
        labels=[0, 1]
    )
)


Accuracy: 0.805
Macro F1: 0.7136

Classification report:
              precision    recall  f1-score   support

CONTINUATION       0.88      0.87      0.88       157
 TOPIC_SHIFT       0.55      0.56      0.55        43

    accuracy                           0.81       200
   macro avg       0.71      0.72      0.71       200
weighted avg       0.81      0.81      0.81       200

Confusion matrix:
[[137  20]
 [ 19  24]]


In [ ]:
import os
import shutil
from google.colab import files

# Save the fine-tuned model and tokenizer
save_dir = "/content/bert_tiage_classifier_2"

model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

# Package everything into a ZIP file
zip_path = shutil.make_archive(
    "/content/bert_tiage_classifier_2",
    "zip",
    root_dir="/content",
    base_dir="bert_tiage_classifier_2"
)

print("Model saved and packaged:", zip_path)

# Download to your computer
files.download(zip_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved and packaged: /content/bert_tiage_classifier_2.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Probabilities for class 1 (TOPIC_SHIFT)
shift_probs = np.asarray(all_probabilities)[:, 1]
y_true = np.asarray(all_labels)

for threshold in [0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]:
    y_pred = (shift_probs >= threshold).astype(int)

    print(f"\nThreshold: {threshold:.2f}")
    print(f"Accuracy:  {accuracy_score(y_true, y_pred):.3f}")
    print(f"Precision: {precision_score(y_true, y_pred, zero_division=0):.3f}")
    print(f"Recall:    {recall_score(y_true, y_pred, zero_division=0):.3f}")
    print(f"Shift F1:  {f1_score(y_true, y_pred, zero_division=0):.3f}")
    print("Confusion matrix:")
    print(confusion_matrix(y_true, y_pred))


Threshold: 0.35
Accuracy:  0.800
Precision: 0.533
Recall:    0.558
Shift F1:  0.545
Confusion matrix:
[[136  21]
 [ 19  24]]

Threshold: 0.40
Accuracy:  0.805
Precision: 0.545
Recall:    0.558
Shift F1:  0.552
Confusion matrix:
[[137  20]
 [ 19  24]]

Threshold: 0.45
Accuracy:  0.805
Precision: 0.545
Recall:    0.558
Shift F1:  0.552
Confusion matrix:
[[137  20]
 [ 19  24]]

Threshold: 0.50
Accuracy:  0.805
Precision: 0.545
Recall:    0.558
Shift F1:  0.552
Confusion matrix:
[[137  20]
 [ 19  24]]

Threshold: 0.55
Accuracy:  0.810
Precision: 0.558
Recall:    0.558
Shift F1:  0.558
Confusion matrix:
[[138  19]
 [ 19  24]]

Threshold: 0.60
Accuracy:  0.810
Precision: 0.558
Recall:    0.558
Shift F1:  0.558
Confusion matrix:
[[138  19]
 [ 19  24]]

Threshold: 0.65
Accuracy:  0.825
Precision: 0.600
Recall:    0.558
Shift F1:  0.578
Confusion matrix:
[[141  16]
 [ 19  24]]

Threshold: 0.70
Accuracy:  0.830
Precision: 0.615
Recall:    0.558
Shift F1:  0.585
Confusion matrix:
[[142  15]
 [ 1

In [ ]:
print("Min:", shift_probs.min())
print("Max:", shift_probs.max())
print("Quantiles:", np.quantile(
    shift_probs, [0, 0.1, 0.25, 0.5, 0.75, 0.9, 1]
))
print("Unique predictions at 0.5:",
      np.unique((shift_probs >= 0.5).astype(int), return_counts=True))

Min: 4.8948958e-05
Max: 0.999894
Quantiles: [4.89489576e-05 5.39251032e-05 5.94999274e-05 8.04022457e-05
 6.62670238e-03 9.99833369e-01 9.99894023e-01]
Unique predictions at 0.5: (array([0, 1]), array([156,  44]))
